# 综合工程实践

学习目标：把文本文件分析做成可测试、可配置、可安装的命令行工具，并验证批处理结果与性能取舍。

前置知识：文件与编码、数据类、异常处理、命令行与日志、pytest、项目打包、线程池和性能分析。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

测试输入、构建副本和安装目标均在临时目录中创建并清理。

配套脚本：位于 [scripts/30-engineering-practice/](scripts/30-engineering-practice/)。

1. [src/study_file_report/core.py](scripts/30-engineering-practice/src/study_file_report/core.py)：文件计数和线程批处理。
2. [src/study_file_report/cli.py](scripts/30-engineering-practice/src/study_file_report/cli.py)、[\_\_main\_\_.py](scripts/30-engineering-practice/src/study_file_report/__main__.py)：参数、配置、日志和程序入口。
3. [tests/](scripts/30-engineering-practice/tests/)：计数、编码、批处理和命令行测试。
4. [pyproject.toml](scripts/30-engineering-practice/pyproject.toml)、[example.toml](scripts/30-engineering-practice/example.toml)：打包配置与应用配置。

## 1 先明确统计口径

本例分析调用者明确给出的 UTF-8 文本文件，不递归扫描目录。一次调用输出一个 JSON 数组，结果顺序与输入顺序相同；重复输入也保留为两条结果。

| 字段 | 中文含义／本例约定 |
| --- | --- |
| path | 输入路径 |
| lines | 文本流逐行读取的次数；空文件为 0 |
| words | str.split() 按空白分隔出的项数，不进行中文分词 |
| characters | Unicode 码位数，包含原始换行字符，不是 UTF-8 字节数 |

newline="" 会识别 LF、CR 和 CRLF，同时保留原换行。末尾一个换行不会凭空再产生一行；CRLF 在 characters 中算两个字符。下面先观察这些输入约定。

In [1]:
from io import StringIO

sample = "Python 学习\r\n第二行"
with StringIO(sample, newline="") as stream:
    lines = list(stream)

print(lines)  # ['Python 学习\r\n', '第二行']
print([len(line.split()) for line in lines])  # [2, 1]：没有中文分词。
print(len(sample), len(sample.encode("utf-8")))  # 14 24：码位数与字节数不同。

['Python 学习\r\n', '第二行']
[2, 1]
14 24


## 2 把计算与程序入口分开

core.py 的 summarize() 只消费文本行；analyze_file() 负责打开和关闭文件，返回 FileStats 数据类。cli.py 再组合参数、配置与输出，避免让统计函数依赖命令行状态。

这里复用前面学过的模块、数据类与迭代器。本章关注它们如何组合成可单独测试的接口。

下面临时把项目 src 加入导入路径，导入后立即恢复搜索路径；实际安装演示放在后面的打包小节。

In [2]:
from pathlib import Path
import sys

project = Path("scripts/30-engineering-practice").resolve()
source_root = project / "src"
previous_bytecode = sys.dont_write_bytecode
sys.path.insert(0, str(source_root))
try:
    sys.dont_write_bytecode = True
    from study_file_report.core import analyze_batch, analyze_file, summarize
    from study_file_report.cli import load_options
finally:
    sys.path.pop(0)
    sys.dont_write_bytecode = previous_bytecode

with StringIO("one two\nthree", newline="") as stream:
    print(summarize(stream))  # (2, 3, 13)
print(summarize([]))  # (0, 0, 0)：空输入有明确结果。

(2, 3, 13)
(0, 0, 0)


## 3 读取文件与定位失败

analyze_file() 在 with 中读取，正常结束或解码失败时都会关闭文件。输入不是有效 UTF-8 时，函数用 ValueError 补充文件路径，并用异常链保留原来的 UnicodeDecodeError。

本例严格拒绝解码错误，不通过忽略字节获得不完整的统计。缺失路径继续抛出 FileNotFoundError，不能当作空文件。

In [3]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    folder = Path(directory)
    path = folder / "notes.txt"
    # 固定字节，避免操作系统在写入阶段转换换行。
    path.write_bytes(b"one two\r\nthree")
    result = analyze_file(path)
    print(result.lines, result.words, result.characters)  # 2 3 14

    bad_path = folder / "bad.txt"
    bad_path.write_bytes(b"\xff")
    try:
        analyze_file(bad_path)
    except ValueError as error:
        print(bad_path.name in str(error))  # True：诊断能定位输入。
        print(type(error.__cause__).__name__)  # UnicodeDecodeError
    else:
        raise AssertionError("错误编码必须被拒绝")

print(folder.exists())  # False：关闭文件后临时目录可以清理。

2 3 14
True
UnicodeDecodeError
False


## 4 通过命令行使用同一个功能

命令行入口接收一个或多个文件路径。--workers 设置线程数，--config 指定 TOML 配置，--log-level 调整日志级别。

开发时可用 python -m study_file_report；安装后还会提供 study-file-report 命令。下面用子进程运行真实入口，PYTHONPATH 只设置给子进程，不依赖前面的 Notebook 导入状态。

run_source() 仅方便本章重复调用；它返回 CompletedProcess，让示例分别检查标准输出、标准错误和退出状态。

In [4]:
import os
import subprocess


def run_source(
    arguments: list[str], *, cwd: Path
) -> subprocess.CompletedProcess[str]:
    """在指定目录通过源码运行真实命令行入口。"""
    # 给子进程指定源码搜索路径；cwd 控制传入文件的相对路径起点。
    environment = dict(
        os.environ,
        PYTHONPATH=str(source_root),
        PYTHONDONTWRITEBYTECODE="1",
        PYTHONIOENCODING="utf-8",
    )
    return subprocess.run(
        [sys.executable, "-B", "-m", "study_file_report", *arguments],
        cwd=cwd, env=environment,
        capture_output=True, text=True, encoding="utf-8", timeout=30,
    )


help_result = run_source(["--help"], cwd=project)
assert help_result.returncode == 0
print(help_result.stdout.strip())  # usage 包含 paths、--config、--workers、--log-level 及各自用途。

usage: study-file-report [-h] [--config CONFIG] [--workers WORKERS]
                         [--log-level {DEBUG,INFO,WARNING,ERROR,CRITICAL}]
                         paths [paths ...]

统计 UTF-8 文本文件

positional arguments:
  paths                 输入文件路径

options:
  -h, --help            show this help message and exit
  --config CONFIG       TOML 配置文件
  --workers WORKERS     覆盖配置中的线程数
  --log-level {DEBUG,INFO,WARNING,ERROR,CRITICAL}
                        覆盖配置中的日志级别


### 4.1 结果与诊断分开输出

成功时，标准输出只有完整 JSON；日志进入标准错误。工具先收集所有结果，再输出一个数组，避免某个输入失败时留下半份结果。

这保证本例在分析阶段失败时没有部分 JSON，不表示操作系统写入标准输出时具有事务性。

In [5]:
import json

with TemporaryDirectory() as directory:
    folder = Path(directory)
    (folder / "a.txt").write_bytes(b"one two\n")
    (folder / "b.txt").write_bytes(b"three")
    completed = run_source(["a.txt", "b.txt"], cwd=folder)
    assert completed.returncode == 0, completed.stderr
    print(json.loads(completed.stdout))
    # a.txt：1 行、2 项、8 个字符；b.txt：1 行、1 项、5 个字符。
    print(completed.stderr == "")  # True：默认级别下没有普通完成日志。

[{'path': 'a.txt', 'lines': 1, 'words': 2, 'characters': 8}, {'path': 'b.txt', 'lines': 1, 'words': 1, 'characters': 5}]
True


## 5 加入配置与覆盖规则

应用配置放在 example.toml 的 analysis 表中，当前只接受 workers 和 log_level。pyproject.toml 是项目与工具配置，两者职责不同。

本例约定优先级为：默认值 → 合法配置文件 → 显式命令行选项。配置文件先完整校验，即使命令行覆盖某个选项，也不允许配置中保留拼写错误或非法值。

workers 必须是正整数，拒绝 bool；未知表名、未知选项和不支持的日志级别都作为配置错误。

In [6]:
print((project / "example.toml").read_text(encoding="utf-8").strip())  # 显示 [analysis] 段：workers = 2、log_level = "INFO"。
print(load_options(None))  # (1, 'WARNING')：没有配置时的默认值。
print(load_options(project / "example.toml"))  # (2, 'INFO')

with TemporaryDirectory() as directory:
    folder = Path(directory)
    (folder / "notes.txt").write_bytes(b"one")
    completed = run_source(
        ["notes.txt", "--config", str(project / "example.toml"),
         "--workers", "1"],
        cwd=folder,
    )
    assert completed.returncode == 0, completed.stderr
    print(completed.stderr.strip())  # INFO: 完成 1 个文件，workers=1
    print(json.loads(completed.stdout)[0]["words"])  # 1

[analysis]
workers = 2
log_level = "INFO"
(1, 'WARNING')
(2, 'INFO')


INFO: 完成 1 个文件，workers=1
1


## 6 用退出状态表示失败

| 退出状态 | 中文含义／本例处理 |
| --- | --- |
| 0 | 成功，或用户请求帮助 |
| 1 | 配置校验、文件读取或解码失败 |
| 2 | argparse 发现命令行语法错误 |

main() 只在程序边界处理预期的 OSError 和 ValueError，向标准错误写出诊断，且不受日志级别过滤；库函数保留异常，便于其他调用者决定怎么处理。

logger 的处理器在调用结束后移除并关闭，同时恢复级别和传播设置，避免反复调用 main() 时累计重复日志。

In [7]:
with TemporaryDirectory() as directory:
    folder = Path(directory)
    (folder / "good.txt").write_bytes(b"ok")
    (folder / "bad.txt").write_bytes(b"\xff")

    failed = run_source(["good.txt", "bad.txt"], cwd=folder)
    print(failed.returncode, failed.stdout == "")  # 1 True
    print(failed.stderr.strip())  # 分析失败：bad.txt: 输入不是 UTF-8 文本。

    malformed = run_source(
        ["good.txt", "--workers", "many"], cwd=folder
    )
    print(malformed.returncode, malformed.stdout == "")  # 2 True
    assert "many" in malformed.stderr

1 True
分析失败：bad.txt: 输入不是 UTF-8 文本


2 True


## 7 加入批处理并保持结果一致

analyze_batch() 在线程数为 1 时顺序调用；大于 1 时用 ThreadPoolExecutor.map()。map 按输入顺序提供结果，因此无需让工作线程修改共享结果列表。

文件由每次 analyze_file() 调用独立打开，线程池通过 with 关闭。取结果时出现异常会传播；退出线程池仍会等待已提交的工作完成，不等于强制停止所有读取。

本例适合数量有限、内容稳定的普通文件。Python 3.12 的 Executor.map 会立即收集并提交输入，不能把它当作处理无限任务的有界队列。

In [8]:
with TemporaryDirectory() as directory:
    folder = Path(directory)
    first = folder / "first.txt"
    second = folder / "second.txt"
    first.write_bytes(b"one two")
    second.write_bytes(b"three")
    paths = [second, first, second]

    sequential = analyze_batch(paths, workers=1)
    threaded = analyze_batch(paths, workers=2)
    print(threaded == sequential)  # True：相同统计与相同顺序。
    print([item.words for item in threaded])  # [1, 2, 1]
    print([Path(item.path).name for item in threaded])
    # ['second.txt', 'first.txt', 'second.txt']：重复输入仍保留。

True
[1, 2, 1]
['second.txt', 'first.txt', 'second.txt']


## 8 让测试覆盖约定与边界

tests/test_core.py 检查空文件、不同换行、中文码位、错误编码和批处理顺序；tests/test_cli.py 检查 JSON 输出、配置覆盖、重复调用与错误退出。

测试使用实际临时文件和真实入口，不把核心计算替换成 mock。期望值由小输入直接确定，而不是再调用同一统计函数生成答案。

下面在子进程运行 pytest，避免 Notebook 中已导入的模块影响测试发现。测试临时目录随 with 清理，关闭 pytest 缓存。

In [9]:
with TemporaryDirectory() as directory:
    test_result = subprocess.run(
        [sys.executable, "-B", "-m", "pytest", "-q",
         "-p", "no:cacheprovider", "--basetemp", directory],
        cwd=project,
        env=dict(
            os.environ, PYTHONPATH=str(source_root),
            PYTHONDONTWRITEBYTECODE="1",
            PYTEST_DISABLE_PLUGIN_AUTOLOAD="1",
            PYTHONIOENCODING="utf-8",
        ),
        capture_output=True, text=True, encoding="utf-8", timeout=60,
    )
    print(test_result.stdout.strip())  # 30 passed；运行耗时随机器与负载变化。
    assert test_result.returncode == 0, test_result.stderr
    assert not test_result.stderr, test_result.stderr

..............................                                           [100%]
30 passed in 0.63s


### 8.1 在构建前检查代码

pyproject.toml 配置 Ruff 的目标版本、行宽和检查规则。代码检查关注未使用名称、导入与语法等问题；格式检查关注统一排版，均不能替代行为测试。

构建前先运行这些检查，可以尽早发现打包本身不会报告的问题。

In [10]:
for command in (
    ["check", "--no-cache", "."],
    ["format", "--no-cache", "--check", "."],
):
    checked = subprocess.run(
        [sys.executable, "-m", "ruff", *command],
        cwd=project, capture_output=True, text=True,
        encoding="utf-8", timeout=30,
    )
    print(checked.stdout.strip())  # 依次为 All checks passed!、7 files already formatted。
    assert checked.returncode == 0, checked.stdout + checked.stderr

All checks passed!


7 files already formatted


## 9 构建并安装同一个工具

使用 python -m build 构建 sdist 与 wheel，再用 pip install --target 将本地 wheel 安装到临时目录。--no-isolation 使用已准备好的构建依赖，--no-index 与 --no-deps 避免安装时查询索引或解析其他依赖。

下面依次完成构建、安装、模块调用和命令入口加载。工作目录离开源码目录，子进程的 PYTHONPATH 指向安装目标；它仍会保留默认搜索路径，因此还观察实际导入文件是否来自安装目标。临时目录在全部子进程退出后自动清理。

In [11]:
import shutil
import textwrap

with TemporaryDirectory() as directory:
    root = Path(directory).resolve()
    copied = shutil.copytree(project, root / "source")
    dist = root / "dist"
    target = root / "installed"
    env = dict(os.environ, PYTHONDONTWRITEBYTECODE="1", PYTHONIOENCODING="utf-8")

    # 1. 在源码副本中构建，失败直接由 subprocess.run 传播。
    subprocess.run(
        [sys.executable, "-B", "-m", "build", "--no-isolation",
         "--outdir", str(dist), str(copied)],
        env=env, capture_output=True, encoding="utf-8", check=True, timeout=120,
    )
    wheel, = dist.glob("*.whl")
    print(sorted(path.name for path in dist.iterdir()))  # study_file_report 的 0.1.0 wheel 与 sdist，按文件名排序。

    # 2. 把本地 wheel 安装到临时目标，不修改共享环境中的包。
    subprocess.run(
        [sys.executable, "-B", "-m", "pip", "--isolated", "install",
         "--no-index", "--no-deps", "--no-compile", "--no-cache-dir",
         "--target", str(target), str(wheel)],
        env=env, capture_output=True, encoding="utf-8", check=True, timeout=60,
    )

    # 3. 在源码目录外运行；PYTHONPATH 只给当前子进程增加安装目标。
    (root / "input.txt").write_bytes(b"one two\n")
    env["PYTHONPATH"] = str(target)
    result = subprocess.run(
        [sys.executable, "-B", "-m", "study_file_report", "input.txt"],
        cwd=root, env=env, capture_output=True,
        encoding="utf-8", check=True, timeout=30,
    )
    print(json.loads(result.stdout)[0]["words"])  # 2。

    # 4. 读取安装元数据并加载登记的入口，观察它与 -m 使用同一个功能。
    code = textwrap.dedent("""
        import sys
        from importlib import metadata
        from pathlib import Path
        import study_file_report

        # 确认导入来自临时安装目标，避免误用另一个同名包。
        target = Path(sys.argv[1]).resolve()
        assert Path(study_file_report.__file__).resolve().is_relative_to(target)
        dist = metadata.distribution("study-file-report")
        entry, = dist.entry_points.select(
            group="console_scripts", name="study-file-report"
        )
        sys.exit(entry.load()(["input.txt"]))
    """)
    result = subprocess.run(
        [sys.executable, "-B", "-c", code, str(target)],
        cwd=root, env=env, capture_output=True,
        encoding="utf-8", check=True, timeout=30,
    )
    print(json.loads(result.stdout)[0]["words"])  # 2：登记的入口也完成统计。

print(root.exists())  # False：源码副本、安装目标与构建产物均已清理。

['study_file_report-0.1.0-py3-none-any.whl', 'study_file_report-0.1.0.tar.gz']


2


2
False


## 10 比较实现时同时检查正确性与成本

### 10.1 一次读完与逐行统计

另一种直接写法是先 readlines()，再统计全部行。它与逐行版使用相同的换行规则，但会先保留完整的行列表。

下面固定输入，先确认两种写法结果相同，再用 timeit.repeat() 测量。每组执行 5 次，重复 3 组，展示每次调用的秒数；文件缓存、后台负载都会影响数值，不预设哪种一定更快。

In [12]:
import timeit


def count_loaded(path: Path) -> tuple[int, int, int]:
    """先保留全部文本行，再按相同口径统计。"""
    with path.open(encoding="utf-8", newline="") as stream:
        return summarize(stream.readlines())


def count_streamed(path: Path) -> tuple[int, int, int]:
    """使用项目的逐行读取实现，取出三个统计值。"""
    result = analyze_file(path)
    return result.lines, result.words, result.characters


with TemporaryDirectory() as directory:
    benchmark_path = Path(directory) / "benchmark.txt"
    benchmark_path.write_bytes(b"one two three\n" * 20000)
    # 先核对两种读法的统计口径，再计时；生成输入文件的时间不计入。
    expected = (20000, 60000, 280000)
    assert count_loaded(benchmark_path) == expected
    assert count_streamed(benchmark_path) == expected

    for name, operation in (
        ("一次读完", count_loaded), ("逐行统计", count_streamed)
    ):
        samples = timeit.repeat(
            lambda: operation(benchmark_path), repeat=3, number=5
        )
        print(name, [round(sample / 5, 6) for sample in samples])
    # 每个数是一组测量折算出的秒/次；只代表这个固定工作负载。

一次读完 [0.017546, 0.013017, 0.014643]


逐行统计 [0.016375, 0.018116, 0.013529]


### 10.2 查看 Python 分配的内存峰值

tracemalloc 跟踪 Python 分配的内存，不是整个进程的物理内存占用。下面在相同输入上单独跟踪两种实现，并在每次观察后停止跟踪。

逐行统计避免保存全部行，但内存仍受最长单行和该行 split() 结果影响；线程批处理还会同时保留多个文件的工作状态和结果，不能概括为恒定内存。

In [13]:
from collections.abc import Callable
import tracemalloc


def peak_bytes(operation: Callable[[], object]) -> int:
    """测量本次调用的 Python 分配峰值，使用本次独立跟踪会话。"""
    # 从空内核运行，确保此处没有其他跟踪会话。
    tracemalloc.start()
    try:
        operation()
        return tracemalloc.get_traced_memory()[1]
    finally:
        tracemalloc.stop()


with TemporaryDirectory() as directory:
    benchmark_path = Path(directory) / "benchmark.txt"
    benchmark_path.write_bytes(b"one two three\n" * 20000)
    loaded_peak = peak_bytes(lambda: count_loaded(benchmark_path))
    streamed_peak = peak_bytes(lambda: count_streamed(benchmark_path))
    print("一次读完的峰值字节数：", loaded_peak)
    print("逐行统计的峰值字节数：", streamed_peak)
    # 峰值来自本次实际运行；不把固定内存阈值作为正确性标准。

一次读完的峰值字节数： 1430134
逐行统计的峰值字节数： 25786


### 10.3 线程数量需要按实际文件测试

线程池适合尝试重叠文件等待，但小文件、缓存命中和计数计算都会影响收益。先保证顺序版和线程版输出相等，再比较耗时；更多线程不保证更快。

下面只测 8 个固定大小的本地文件。输入在测量期间保持不变，否则两种执行时机可能读取到不同内容。

In [14]:
with TemporaryDirectory() as directory:
    folder = Path(directory)
    paths = []
    for index in range(8):
        path = folder / f"{index}.txt"
        path.write_bytes(b"one two three\n" * 2000)
        paths.append(path)

    reference = analyze_batch(paths, workers=1)
    for workers in (1, 2, 4):
        assert analyze_batch(paths, workers=workers) == reference
        samples = timeit.repeat(
            lambda: analyze_batch(paths, workers=workers),
            number=2, repeat=3,
        )
        print(workers, [round(value / 2, 6) for value in samples])
    # 每行是线程数与三组秒/次；用测量结果决定这个工作负载的设置。

1 [0.010169, 0.010143, 0.011023]
2 [0.01283, 0.01212, 0.011792]
4 [0.012842, 0.012195, 0.01312]


## 本章小结

（1）先定义编码、换行和计数口径，再划分计算、文件读取与程序入口。

（2）配置在边界校验，显式参数覆盖合法配置；结果、诊断和退出状态各有职责。

（3）测试覆盖普通输入与失败行为，Ruff 检查代码，构建后还要在源码目录之外检查安装结果。

（4）批处理保持结果顺序并管理资源；比较性能时先验证正确性，同时考虑耗时、内存和输入规模。

自查：换行规则、输入编码或结果顺序改变时，哪些测试应该失败？为什么增加线程之前需要保留顺序版作为对照？

## 练习

（1）先预测下面文件的三个统计值，再运行。说明字符数为何包含空白与换行，为什么最后一个换行不会增加一条空行。

In [15]:
with TemporaryDirectory() as directory:
    exercise_path = Path(directory) / "example.txt"
    exercise_path.write_bytes("中文\r\nA B\n".encode("utf-8"))
    exercise_result = analyze_file(exercise_path)
    print(
        exercise_result.lines,
        exercise_result.words,
        exercise_result.characters,
    )
    # 运行后按计数口径核对自己的预测。

2 3 8


（2）在项目副本中增加 --min-words 选项，只输出 words 不小于阈值的文件结果，默认阈值为 0；拒绝负数。

先为临界值、空结果数组及过滤后顺序编写测试，再修改实现。过滤只能改变输出项，不能把读取失败的文件当作“被过滤掉”而忽略错误。

In [16]:
# 在临时项目副本中增加测试与选项；不要覆盖输入文件。
# 检查阈值 0、恰好等于词项数、全部被过滤、负数和文件读取失败。

（3）在项目副本中提供“继续处理其他文件”的可选模式，输出成功结果和失败列表，并规定只要出现失败就返回非零状态。

用三个输入检查中间文件解码失败时的行为：两个成功文件仍按原顺序输出；失败记录包含路径和原因；顺序版与线程版结果一致。先写清楚新输出格式，再测试，避免把新行为混入现有“整批成功才输出”的接口。

In [17]:
# 在项目副本中实现新的可选模式，原默认行为的测试应继续通过。
# 重新构建 wheel，并从安装结果运行包含失败文件的批处理。

## 参考与引用来源

| 网站 | 资料与知识点定位 |
| --- | --- |
| Python 官方文档（3.12） | [文本打开、编码与 newline 规则](https://docs.python.org/3.12/library/functions.html#open)、[文本流迭代与关闭](https://docs.python.org/3.12/library/io.html#io.IOBase)、[StringIO](https://docs.python.org/3.12/library/io.html#io.StringIO)、[空白分隔](https://docs.python.org/3.12/library/stdtypes.html#str.split)、[数据类](https://docs.python.org/3.12/library/dataclasses.html#dataclasses.dataclass)、[数据类转字典](https://docs.python.org/3.12/library/dataclasses.html#dataclasses.asdict)、[argparse](https://docs.python.org/3.12/library/argparse.html#module-argparse)、[TOML 读取](https://docs.python.org/3.12/library/tomllib.html#tomllib.load)、[Logger 与处理器](https://docs.python.org/3.12/library/logging.html#logger-objects)、[subprocess.run](https://docs.python.org/3.12/library/subprocess.html#subprocess.run)、[Executor.map](https://docs.python.org/3.12/library/concurrent.futures.html#concurrent.futures.Executor.map)、[执行器关闭](https://docs.python.org/3.12/library/concurrent.futures.html#concurrent.futures.Executor.shutdown)、[入口元数据](https://docs.python.org/3.12/library/importlib.metadata.html#entry-points)、[发行包名称与版本](https://docs.python.org/3.12/library/importlib.metadata.html#distributions)、[发行包文件清单](https://docs.python.org/3.12/library/importlib.metadata.html#distribution-files)、[PYTHONPATH 扩展默认搜索路径](https://docs.python.org/3.12/using/cmdline.html#envvar-PYTHONPATH)、[模块文件来源](https://docs.python.org/3.12/reference/datamodel.html#module.__file__)、[函数所属模块](https://docs.python.org/3.12/reference/datamodel.html#function.__module__)、[已加载模块](https://docs.python.org/3.12/library/sys.html#sys.modules)、[路径解析](https://docs.python.org/3.12/library/pathlib.html#pathlib.Path.resolve)、[路径归属判断](https://docs.python.org/3.12/library/pathlib.html#pathlib.PurePath.is_relative_to)、[多行代码去除公共缩进](https://docs.python.org/3.12/library/textwrap.html#textwrap.dedent)、[timeit.repeat](https://docs.python.org/3.12/library/timeit.html#timeit.repeat)、[tracemalloc 峰值](https://docs.python.org/3.12/library/tracemalloc.html#tracemalloc.get_traced_memory)、[TemporaryDirectory](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryDirectory)。字段、配置优先级及整批失败策略是本例自行规定的应用接口。 |
| pytest 文档 | pytest 9.1.1：[断言与预期异常](https://docs.pytest.org/en/stable/how-to/assert.html#assertions-about-expected-exceptions)、[参数化](https://docs.pytest.org/en/stable/how-to/parametrize.html)、[临时目录](https://docs.pytest.org/en/stable/how-to/tmp_path.html)、[输出捕获](https://docs.pytest.org/en/stable/how-to/capture-stdout-stderr.html)。 |
| Python Packaging User Guide | [项目打包教程](https://packaging.python.org/en/latest/tutorials/packaging-projects/)、[pyproject.toml 与命令行入口](https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#creating-executable-scripts)。 |
| setuptools 文档 | [pyproject.toml 配置](https://setuptools.pypa.io/en/latest/userguide/pyproject_config.html)，本例后端固定为 setuptools 83.0.0。 |
| build 文档 | [命令行构建与 --no-isolation](https://build.pypa.io/en/stable/)，本例使用 build 1.6.1。 |
| pip 文档 | [--target 安装目录](https://pip.pypa.io/en/stable/cli/pip_install/#cmdoption-t)、[本地分发包安装示例](https://pip.pypa.io/en/stable/cli/pip_install/#examples)。 |
| Ruff 文档 | [项目配置、目标版本与检查规则](https://docs.astral.sh/ruff/configuration/)，本例使用 Ruff 0.16.7。 |
| GitHub（CPython 官方源码） | [Python 3.12 的 importlib.metadata](https://github.com/python/cpython/blob/3.12/Lib/importlib/metadata/__init__.py)：Distribution.files 与 Distribution.locate_file 的公开接口定义。 |